# Aula 2 · Do dado ao modelo (texto + tabela, juntos)

Aqui montamos o modelo **de verdade**: um único modelo que usa o **texto** da avaliação
**e** os dados de **tabela** (preço, frete, prazo) ao mesmo tempo.

### Por que juntar texto e tabela?

Só **~40%** dos clientes escrevem um comentário. Um modelo só de texto seria **cego**
para os outros 60%. Juntando as duas fontes:

- quando **tem** texto, o modelo usa o texto (mais informativo);
- quando **não tem**, ele se vira com a tabela (preço, prazo, atraso...).

Assim ele prevê para **todos** os pedidos. Esse é o jeito que se faz na vida real.

> Pré-requisito: ter visto o notebook `00-como-texto-vira-numero-tfidf` (como o texto vira numero).

### O caminho deste notebook
`coletar → tratar → separar (holdout) → treinar → VALIDAR (com calma) → salvar`

## 1. Pegar os dados — TODOS os pedidos

Diferente do modelo só de texto: aqui pegamos todo mundo. Quem não escreveu
comentário fica com o texto vazio (`""`).

In [ ]:
import urllib.request, pathlib
import pandas as pd

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("olist.parquet")
if not PATH.exists():
    print("Baixando..."); urllib.request.urlretrieve(URL, PATH)

df = pd.read_parquet(PATH)
print(f"{len(df):,} pedidos no total")

tem = df["review_comment_message"].fillna("").str.len() > 0
print(f"Com comentario:  {tem.sum():,} ({tem.mean()*100:.0f}%)")
print(f"Sem comentario:  {(~tem).sum():,} ({(~tem).mean()*100:.0f}%)")

## 2. Montar as features e o rótulo

**Feature** = o que o modelo usa para decidir. **Rótulo** = a resposta certa.

| Tipo | Colunas |
|------|---------|
| Texto | o comentário (vazio quando não existe) |
| Numéricas | preço, frete, dias de entrega, dias estimados, atrasou |
| Categóricas | estado, categoria do produto |
| **Rótulo** | satisfeito = nota ≥ 4 |

In [ ]:
dados = pd.DataFrame({
    "texto": df["review_comment_message"].fillna("").str.lower().str.replace(r"[^a-zà-ú0-9 ]", "", regex=True),
    "preco": df["price"],
    "frete": df["freight_value"],
    "dias_entrega": df["delivery_days"],
    "dias_estimado": df["estimated_delivery_days"],
    "atrasou": df["delivered_late"].astype("Int64").fillna(0),
    "estado": df["customer_state"],
    "categoria": df["product_category_en"].fillna("desconhecida"),
})
dados["satisfeito"] = (df["review_score"] >= 4).astype(int)
dados["tem_texto"] = (dados["texto"].str.len() > 0).astype(int)
dados = dados.dropna(subset=["dias_entrega"])
print(f"{len(dados):,} pedidos prontos")
dados.head()

## 3. Esconder uma parte para testar (holdout)

Como saber se o modelo **aprendeu** ou só **decorou**? Escondemos 20% dos dados.
Treinamos nos 80% e testamos nos 20% escondidos — dados que ele **nunca viu**.

> `random_state=42` faz a divisão ser sempre igual, para o resultado ser reproduzível.

In [ ]:
from sklearn.model_selection import train_test_split

X = dados.drop(columns=["satisfeito"])
y = dados["satisfeito"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Treino: {len(X_tr):,}  |  Teste (escondido): {len(X_te):,}")

## 4. Montar o modelo combinado

O `ColumnTransformer` aplica o tratamento certo em cada tipo de coluna, tudo de uma vez:

- **texto** → `TfidfVectorizer` (transforma em números, como vimos no notebook 00)
- **numéricas** → padroniza a escala
- **categóricas** → vira colunas 0/1 (one-hot)

Quando o texto é vazio, o TF-IDF vira só zeros e o modelo **se apoia na tabela**.
**Sem `if`, sem escolher** — ele lida com isso sozinho.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

NUM = ["preco", "frete", "dias_entrega", "dias_estimado", "atrasou"]
CAT = ["estado", "categoria"]

preproc = ColumnTransformer([
    ("texto", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3), "texto"),
    ("num",   Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
    ("cat",   OneHotEncoder(handle_unknown="ignore", min_frequency=30), CAT),
])

modelo = Pipeline([
    ("features", preproc),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
modelo.fit(X_tr, y_tr)
print("Modelo combinado treinado!")

---

# 5. Validar o modelo (com calma)

Um número só — a acurácia — **engana**. Como a maioria das avaliações é positiva,
um modelo preguiçoso que chuta "satisfeito" sempre já acertaria bastante.

Por isso vamos olhar o modelo de **4 ângulos**:
1. Onde ele cobre (com e sem texto)
2. Onde ele erra (matriz de confusão)
3. As 3 notas (acurácia, precisão, recall)
4. O quão confiante ele é (probabilidades) e a curva ROC

## 5.1 — Ele cobre todo mundo?

A grande sacada do modelo combinado: funciona para quem escreveu **e** para quem não escreveu.

In [ ]:
from sklearn.metrics import accuracy_score

pred = modelo.predict(X_te)
proba = modelo.predict_proba(X_te)[:, 1]   # probabilidade de 'satisfeito'
tem_te = X_te["tem_texto"].values

print(f"Acuracia GERAL (todos):  {accuracy_score(y_te, pred)*100:.1f}%")
print(f"  - pedidos COM texto:   {accuracy_score(y_te[tem_te==1], pred[tem_te==1])*100:.1f}%")
print(f"  - pedidos SEM texto:   {accuracy_score(y_te[tem_te==0], pred[tem_te==0])*100:.1f}%")

Repare: o modelo vai bem nos **dois** grupos. Usa o texto como bônus quando existe,
e se apoia na tabela quando não existe.

| Abordagem | Acurácia | Cobertura |
|-----------|----------|-----------|
| Só texto | ~89% | só 40% dos pedidos |
| Só tabela | ~73% | todos |
| **Combinado** | **~86%** | **todos** |

**A lição: não escolha entre texto e tabela — use os dois.**

## 5.2 — Onde ele erra? (matriz de confusão)

A matriz mostra os 4 casos possíveis: acertos na diagonal, erros fora dela.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_te, pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Purples")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                fontsize=15, fontweight="bold",
                color="white" if cm[i,j] > cm.max()/2 else "black")
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["previu NEG", "previu POS"])
ax.set_yticklabels(["era NEG", "era POS"])
ax.set_title("Matriz de confusao")
plt.tight_layout(); plt.show()

Ele acerta muito mais os **satisfeitos** (maioria) do que os **insatisfeitos**
(minoria) — é o efeito de a base ser desbalanceada. Isso é honesto mostrar.

## 5.3 — As 3 notas: acurácia, precisão, recall

Três perguntas diferentes sobre o mesmo modelo:

- **Acurácia:** dos pedidos todos, quantos ele acertou?
- **Precisão:** quando ele diz "satisfeito", quantas vezes acerta?
- **Recall:** dos que eram satisfeitos, quantos ele conseguiu pegar?

In [ ]:
from sklearn.metrics import precision_score, recall_score

acc = accuracy_score(y_te, pred)
prec = precision_score(y_te, pred)
rec = recall_score(y_te, pred)
print(f"Acuracia: {acc*100:.0f}%")
print(f"Precisao: {prec*100:.0f}%  (quando diz 'satisfeito', acerta)")
print(f"Recall:   {rec*100:.0f}%  (dos satisfeitos, quantos pega)")

## 5.4 — O modelo está confiante? (probabilidades)

O modelo não responde só "sim/não": ele dá uma **probabilidade**. Se ele separa
bem os dois grupos, os satisfeitos ficam com probabilidade alta e os insatisfeitos
com probabilidade baixa.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(proba[y_te==1], bins=40, alpha=0.7, color="#2f9e44", label="Era satisfeito")
ax.hist(proba[y_te==0], bins=40, alpha=0.7, color="#e03131", label="Era insatisfeito")
ax.axvline(0.5, color="#333", linestyle="--")
ax.set_xlabel("Probabilidade de 'satisfeito' que o modelo deu")
ax.set_ylabel("Pedidos"); ax.legend()
ax.set_title("O modelo separa os dois grupos?")
plt.tight_layout(); plt.show()

Verde à direita (alta probabilidade), vermelho à esquerda (baixa). A sobreposição
no meio são os **casos difíceis** — onde até um humano hesitaria.

## 5.5 — Curva ROC: a qualidade num gráfico

A ROC resume tudo: quanto mais a curva sobe para o **canto superior esquerdo**, melhor.
A diagonal seria um chute aleatório. A **área embaixo (AUC)** vira um número de
0.5 (aleatório) a 1.0 (perfeito).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_te, proba)
auc = roc_auc_score(y_te, proba)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color="#7048e8", linewidth=3, label=f"Modelo (AUC = {auc:.2f})")
ax.plot([0,1], [0,1], "--", color="#adb5bd", label="Aleatorio (0.50)")
ax.set_xlabel("Falsos alarmes"); ax.set_ylabel("Acertos verdadeiros")
ax.set_title("Curva ROC"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

---

## 6. Testar com exemplos

A prova final: dar ao modelo casos que fazem sentido e ver se ele responde bem.

In [ ]:
def prever(texto, preco=120, frete=15, dias_entrega=10, dias_estimado=14,
           atrasou=0, estado="SP", categoria="electronics"):
    linha = pd.DataFrame([{
        "texto": texto.lower(), "preco": preco, "frete": frete,
        "dias_entrega": dias_entrega, "dias_estimado": dias_estimado,
        "atrasou": atrasou, "estado": estado, "categoria": categoria,
        "tem_texto": int(len(texto) > 0),
    }])
    p = modelo.predict_proba(linha)[0][1]
    print(f"{'POS' if p>=0.5 else 'NEG'} ({p:.0%})  texto='{texto}' | atrasou={atrasou}")

prever("produto excelente, recomendo!")                          # texto positivo
prever("veio quebrado e ninguem resolve", atrasou=1)             # texto negativo
prever("", dias_entrega=40, dias_estimado=15, atrasou=1)         # SEM texto, atrasado -> usa a tabela
prever("", dias_entrega=8, dias_estimado=14, atrasou=0)          # SEM texto, no prazo

## 7. Salvar o modelo e as predições

Fecha o ciclo: **coletar → tratar → treinar → validar → salvar**.
O modelo salvo (`.joblib`) é o mesmo que a API usa para servir no n8n.

In [ ]:
import joblib

joblib.dump(modelo, "modelo_combinado.joblib")

resultado = X_te.copy()
resultado["satisfeito_real"] = y_te.values
resultado["previsao"] = pred
resultado["probabilidade"] = proba.round(3)
resultado.to_parquet("predicoes_combinado.parquet")

print("Salvos: modelo_combinado.joblib + predicoes_combinado.parquet")

## Recapitulando

- Um **único modelo** usa texto **e** tabela → cobre **100%** dos pedidos.
- Validamos de 4 ângulos (cobertura, confusão, métricas, probabilidades/ROC) —
  porque **um número só engana**.
- O modelo salvo vira o serviço que o n8n consome (ver pasta `api-do-modelo/`).

**Na Aula 4:** um agente de IA (LLM) faz essa classificação **sem treinar nada** —
só lendo. Será que ele bate este modelo? 🚀